# Comparing backscatter conventions

This notebook loads Sentinel-1 Normalised Radar Backscatter (NRB) for one area and time, and shows it as gamma0, DEM sigma0, ellipsoid sigma0 and beta0 next to each other.

NRB is delivered as gamma0. The other conventions are calculated from it with the conversion layers that come with each item:

| Convention | Calculation | Reference area |
| --- | --- | ----------- |
| `gamma0` | As delivered | The area perpendicular to the radar look direction, using the DEM. Terrain brightness effects are largely removed |
| `sigma0_dem` | `gamma0 * oa_gamma0_to_sigma0_ratio` | The ground area, including terrain. Slopes facing the radar look brighter |
| `sigma0_ellipsoid` | `gamma0 * oa_gamma0_to_beta0_ratio * sin(oa_incidence_angle)` | The ground area of the ellipsoid, without terrain. No DEM is used, so DEM errors don't show in the image |
| `beta0` | `gamma0 * oa_gamma0_to_beta0_ratio` | The radar slant range. Not normalised for terrain or incidence angle |

The conversions are done with `convert_backscatter` from `de_sar_demo.sar_utils`. To load a single convention directly, use `load_nrb` from `de_sar_demo.loading`. See [backscatter_conventions.md](../../docs/backscatter_conventions.md) for more on each convention.

Every convention goes through the same steps as the [hero images notebook](visualisation_s1_hero_images.ipynb): optional layover and shadow masking, the Lee speckle filter, a composite over time, and a style.
All images use the same style and display ranges, so differences between them come only from the convention.

## Set up

In [ ]:
import os

# Use the environment's own PROJ/GDAL data, not any inherited paths
for _var in ("PROJ_LIB", "PROJ_DATA", "GDAL_DATA"):
    os.environ.pop(_var, None)

from dataclasses import replace
from pathlib import Path

from dask.diagnostics import ProgressBar
import matplotlib.pyplot as plt
import numpy as np
from odc.stac import load, configure_s3_access
import pandas as pd
from pystac_client import Client
import shapely

from locations import LOCATIONS
from de_sar_demo.masking import dilate_mask
from de_sar_demo.sar_utils import REQUIRED_LAYERS, convert_backscatter
from de_sar_demo.speckle_filters import lee_filter_xr
from de_sar_demo.visualisation import STYLES, apply_style, default_style_for_collection, save_image

stac_client = Client.open("https://explorer.dev.dea.ga.gov.au/stac")
configure_s3_access(cloud_defaults=True, aws_unsigned=True)

## Settings

Choose a location from [locations.py](locations.py), and optionally change its resolution or time range.

| Setting | Description |
| --- | ----------- |
| `LOCATION_NAME` | Any name from `LOCATIONS` in [locations.py](locations.py) |
| `STYLE` | A style from the hero images notebook, or `"auto"` to choose from the collection's polarisation. `multitemporal_rgb` shows each convention over time instead of a composite |
| `PERCENTILES` | `None` uses the style's fixed display ranges, so the conventions can be compared directly. Setting percentiles stretches each image separately, which hides overall brightness differences between them |
| `SCALE` | Display the images in decibels (`"db"`) or linear scale (`"linear"`), or `None` for the style's own scale. The difference maps are always in dB |
| `CRS` | The output CRS, or `"auto"` for EPSG:3031 in Antarctica and EPSG:3577 elsewhere |
| `SPECKLE_FILTER` | Whether to apply the Lee speckle filter |
| `SPECKLE_SIZE` | The Lee filter window, in pixels, when `SPECKLE_FILTER` is `True` |
| `COMPOSITE` | How to combine time steps: `"mean"`, `"median"` or `"first"`. The same method is used for every convention |
| `MASK_LAYOVER_SHADOW` | Whether to remove pixels affected by layover and shadow |
| `MASK_DILATION_METRES` | How far to grow the layover and shadow mask, on the ground |

**Tip** - start with a coarse resolution (e.g. 200 m) to check the area, then lower it. This notebook loads four layers per time step, so it needs about four times the memory of a gamma0-only load.

In [ ]:
LOCATION_NAME = "amery_ice_shelf_front"
location = LOCATIONS[LOCATION_NAME]

# Change part of the location, e.g. a coarser resolution for a quick look
# location = replace(location, resolution=200, start_date="2025-01-01", end_date="2025-01-10")

STYLE = "auto"
PERCENTILES = None  # fixed ranges so the conventions can be compared
SCALE = "linear" #location.display_scale  # or None, "db", "linear"
CRS = "auto"
SPECKLE_FILTER = False #location.speckle_filter  # or True, False
SPECKLE_SIZE = 5
COMPOSITE = "median"
MASK_LAYOVER_SHADOW = location.mask_layover_shadow
MASK_DILATION_METRES = 60
OUTPUT_DIR = Path("outputs/backscatter_conventions")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FONT_SIZE = 18  # text size in the comparison plots

CONVENTIONS = ["gamma0", "sigma0_dem", "sigma0_ellipsoid", "beta0"]

## Search for data

In [ ]:
style_name = default_style_for_collection(location.collection) if STYLE == "auto" else STYLE
print(location.description)
print(f"Style: {style_name} - {STYLES[style_name].title}")

aoi_shape = shapely.geometry.shape(location.aoi.polygon)
items = stac_client.search(
    collections=[location.collection],
    datetime=f"{location.start_date}/{location.end_date}",
    intersects=aoi_shape,
).item_collection()

print(f"Found {len(items)} items")
if len(items) == 0:
    raise ValueError("No items found. Check the coverage of this collection, or try a different area or time range.")
display(
    pd.DataFrame(
        {
            "date": item.datetime.date(),
            "orbit": item.properties["sat:absolute_orbit"],
            "direction": item.properties["sat:orbit_state"],
        }
        for item in items
    ).value_counts().rename("bursts").sort_index().to_frame()
)

## Load the data

The backscatter bands depend on the collection's polarisation, so they are read from the first item.
The two conversion ratios and the incidence angle are loaded with them, so every convention can be calculated from the one load.

Over Antarctica, we group by absolute orbit to combine the bursts from each satellite pass.
Elsewhere, we group by solar day.

In [ ]:
is_antarctic = location.aoi.top < -60
output_crs = ("EPSG:3031" if is_antarctic else "EPSG:3577") if CRS == "auto" else CRS
group_by = "sat:absolute_orbit" if is_antarctic else "solar_day"

gamma0_assets = [name for name in items[0].assets if name.endswith("_gamma0")]
mask_asset = "oa_layover_shadow_mask"
# Load every conversion layer the chosen conventions need, so they can all be calculated from one load
conversion_layers = sorted({layer for convention in CONVENTIONS for layer in REQUIRED_LAYERS[convention]})

bands = gamma0_assets + conversion_layers + ([mask_asset] if MASK_LAYOVER_SHADOW else [])

ds = load(
    items,
    bands=bands,
    geopolygon=aoi_shape,
    crs=output_crs,
    resolution=location.resolution,
    groupby=group_by,
    chunks={},
).sortby("time")

gigabytes = sum(ds[band].size * ds[band].dtype.itemsize for band in ds.data_vars) / 1e9
print(f"Image size: {ds.sizes['x']} x {ds.sizes['y']} pixels, {ds.sizes['time']} time steps")
print(f"Memory needed to load: about {gigabytes:.2f} GB")
if gigabytes > 8:
    print("Warning: this is a large load. Consider a coarser resolution, or a smaller area or time range.")

In [ ]:
with ProgressBar():
    ds = ds.compute()

## Convert, mask, filter and combine

Each convention is calculated from gamma0 with `convert_backscatter`, while the data is still linear and unfiltered.
The layover and shadow mask is then applied, followed by the Lee filter and the composite over time, the same way for every convention.

In [ ]:
def convert(convention):
    # Return only the backscatter bands, named <polarisation>_<convention>
    return convert_backscatter(ds, convention)[[name.replace("gamma0", convention) for name in gamma0_assets]]


if MASK_LAYOVER_SHADOW:
    # 1 = shadow, 2 = layover, 3 = shadow and layover
    dilation_radius = round(MASK_DILATION_METRES / location.resolution)
    layover_shadow = dilate_mask(ds[mask_asset].isin([1, 2, 3]), dilation_radius=dilation_radius)


def combine(filtered):
    if STYLES[style_name].uses_time:
        return filtered
    if COMPOSITE == "mean":
        return filtered.mean("time")
    if COMPOSITE == "median":
        return filtered.median("time")
    if COMPOSITE == "first":
        return filtered.isel(time=0, drop=True)
    raise ValueError(f"COMPOSITE must be one of 'mean', 'median' or 'first', got: {COMPOSITE}")


composites = {}
for convention in CONVENTIONS:
    backscatter = convert(convention)
    if MASK_LAYOVER_SHADOW:
        backscatter = backscatter.where(~layover_shadow)
    filtered = backscatter.map(lee_filter_xr, size=SPECKLE_SIZE) if SPECKLE_FILTER else backscatter
    composites[convention] = combine(filtered)

## Make the images

Each convention is styled with the same display ranges and saved as a full resolution PNG.

In [ ]:
images = {}
for convention in CONVENTIONS:
    rgba = apply_style(composites[convention], style_name, percentiles=PERCENTILES, scale=SCALE)
    images[convention] = rgba
    ranges = ", ".join(f"{channel} {low:.4g} to {high:.4g}" for channel, (low, high) in rgba.attrs["scale_ranges"].items())
    options = f"_{rgba.attrs['scale']}" + ("" if SPECKLE_FILTER else "_unfiltered")
    file_name = f"{location.name}_{convention}_{style_name}{options}_{location.start_date}_{location.end_date}_{location.resolution}m.png"
    output_path = save_image(rgba, OUTPUT_DIR / file_name)
    print(f"{convention}: display range ({rgba.attrs['scale']}) {ranges}, saved to {output_path}")

### Side by side

These previews are reduced in size. Open the saved PNGs to see the full resolution images.

In [ ]:
any_image = images[CONVENTIONS[0]]
step = max(1, max(any_image.sizes["x"], any_image.sizes["y"]) // 1500)
aspect = any_image.sizes["y"] / any_image.sizes["x"]


def thin(da):
    return da.isel(x=slice(None, None, step), y=slice(None, None, step))


columns = 2 if aspect < 0.75 else len(CONVENTIONS)
rows = int(np.ceil(len(CONVENTIONS) / columns))
with plt.rc_context({"font.size": FONT_SIZE}):
    fig, axes = plt.subplots(rows, columns, figsize=(7 * columns, 7 * aspect * rows + 0.6 * rows), squeeze=False, layout="constrained")
    for ax, convention in zip(axes.flat, CONVENTIONS):
        thin(images[convention]).plot.imshow(ax=ax, rgb="band", add_labels=False)
        ax.set_title(convention)
    for ax in axes.flat:
        ax.set_aspect("equal")
        ax.set_axis_off()
    fig.suptitle(f"{location.name}, {location.start_date} to {location.end_date}, {style_name}")

    comparison_path = OUTPUT_DIR / f"{location.name}_conventions_{style_name}{options}_{location.start_date}_{location.end_date}_{location.resolution}m_comparison.png"
    fig.savefig(comparison_path, dpi=150, bbox_inches="tight")
    print(f"Saved the comparison to {comparison_path}")
    plt.show()

## Differences from gamma0

Each panel shows how much brighter (red) or darker (blue) a convention is than gamma0, in dB, using the first backscatter band.
These differences come from the conversion layers, so they show where the terrain and incidence angle change the result.
Gamma0 and DEM sigma0 both use the DEM, so DEM features, including DEM errors, appear in every panel: `sigma0_ellipsoid` doesn't use the DEM, so its difference shows the DEM's effect on gamma0.
A sharp line or step that doesn't follow anything in the backscatter images is likely to come from the DEM rather than the surface.

The table summarises the backscatter of each convention and its difference from gamma0 over the area.

In [ ]:
band_name = gamma0_assets[0].replace("_gamma0", "")


def to_db(da):
    with np.errstate(divide="ignore", invalid="ignore"):
        return 10 * np.log10(da.where(da > 0))


def single_band(convention):
    da = composites[convention][f"{band_name}_{convention}"]
    return da.median("time") if "time" in da.dims else da


reference = to_db(single_band("gamma0"))
compared = [convention for convention in CONVENTIONS if convention != "gamma0"]
differences = {convention: to_db(single_band(convention)) - reference for convention in compared}

limit = max(float(np.abs(difference).quantile(0.99)) for difference in differences.values()) or 0.1
with plt.rc_context({"font.size": FONT_SIZE}):
    fig, axes = plt.subplots(1, len(compared), figsize=(7 * len(compared), 7 * aspect + 1), squeeze=False)
    for ax, convention in zip(axes.flat, compared):
        thin(differences[convention]).plot.imshow(
            ax=ax, cmap="RdBu_r", vmin=-limit, vmax=limit, add_labels=False, cbar_kwargs={"label": "dB", "shrink": 0.8}
        )
        ax.set_title(f"{convention} minus gamma0")
        ax.set_aspect("equal")
        ax.set_axis_off()
    fig.tight_layout()

    difference_path = OUTPUT_DIR / f"{location.name}_conventions_minus_gamma0{'' if SPECKLE_FILTER else '_unfiltered'}_{location.start_date}_{location.end_date}_{location.resolution}m.png"
    fig.savefig(difference_path, dpi=150, bbox_inches="tight")
    print(f"Saved the difference maps to {difference_path}")
    plt.show()

summary = pd.DataFrame(
    {
        convention: {
            "median (dB)": float(to_db(single_band(convention)).median()),
            "1st percentile (dB)": float(to_db(single_band(convention)).quantile(0.01)),
            "99th percentile (dB)": float(to_db(single_band(convention)).quantile(0.99)),
            "median difference from gamma0 (dB)": 0.0 if convention == "gamma0" else float(differences[convention].median()),
        }
        for convention in CONVENTIONS
    }
).T.round(2)
summary